# Survival Analysis with Progenetix CNV Data

In this notebook, we build a simple survival analysis example using Progenetix data.

The basic workflow:

1. retrieve sample-level metadata,
2. retreive individual-level data with survival information,
3. retrieve sample-level CNV records,
4. convert CNV records into simple gene-level features,
5. and apply two standard survival analysis methods.

We will use:

- **Kaplan–Meier curves** to compare survival between two groups,
- and a **Cox proportional hazards model** to estimate how a CNV-derived feature is associated with hazard.



## Environment Setup

```bash
conda install -c conda-forge scikit-survival
pip install scikit-survival

In [ ]:
import requests
import json
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import io

from tqdm.auto import tqdm

from sksurv.util import Surv
from sksurv.nonparametric import kaplan_meier_estimator
from sksurv.linear_model import CoxPHSurvivalAnalysis
from sksurv.metrics import concordance_index_censored

## Download sample metadata from Progenetix

We begin with the Progenetix `sampletable` service.

This service returns sample annotations in tabular form and supports query filters.
For example, the Progenetix documentation shows that TCGA samples can be retrieved with:

- `filters=pgx:cohort-TCGAcancers`

and that:

- `limit=0` means no result limit.


In [ ]:
# -----------------------------
# Cohort definition
# -----------------------------
TARGET_TCGA_PROJECT = "pgx:TCGA-LUAD"   

# -----------------------------
# Progenetix sampletable endpoint
# -----------------------------
SAMPLETABLE_URL = "https://progenetix.org/services/sampletable/"

In [ ]:
def fetch_sampletable(filters, limit=0, timeout=120):
    # Download a Progenetix sample table as a pandas DataFrame.
    params = {
        "filters": filters,
        "limit": limit,
    }

    response = requests.get(SAMPLETABLE_URL, params=params, timeout=timeout)
    response.raise_for_status()

    print("Status:", response.status_code)
    print("Final URL:", response.url)
    print("Content-Type:", response.headers.get("Content-Type"))

    text = response.text
    df = pd.read_csv(io.StringIO(text), sep="\t")
    return df

In [ ]:
sample_table_df = fetch_sampletable(filters=TARGET_TCGA_PROJECT, limit=0)

print("Shape:", sample_table_df.shape)
print("Columns:")
print(sample_table_df.columns.tolist())
sample_table_df.head()

In [ ]:
# Columns we want to keep from the sample table
BIOSAMPLE_COLUMNS = [
    "biosample_id",
    "individual_id",
    "biosample_name",
    "notes",
    "histological_diagnosis_id",
    "histological_diagnosis_label",
    "pathological_stage_id",
    "pathological_stage_label",
    "sample_origin_type_id",
    "sample_origin_type_label",
    "sampled_tissue_id",
    "sampled_tissue_label",
    "tcgaproject_id",
    "tcgaproject_label",
    "icdo_morphology_id",
    "icdo_topography_id",
    "icdo_topography_label",
]

BIOSAMPLE_COLUMNS = [c for c in BIOSAMPLE_COLUMNS if c in sample_table_df.columns]

biosample_df = sample_table_df[BIOSAMPLE_COLUMNS].copy()

print("Biosample-level metadata shape:", biosample_df.shape)
biosample_df.head()

## Build survival metadata by combining biosample- and individual-level information

The Progenetix sample table is useful for cohort selection and biosample-level metadata, but survival-related information is not always available there.

In our case, each biosample is linked to an `individual_id`.  
We therefore use a two-step strategy:

1. retrieve **biosample-level metadata** from the Progenetix sample table,
2. retrieve **individual-level follow-up metadata** from the Progenetix individual endpoint,
3. and merge the two through `individual_id`.

This is important because survival analysis requires at least:

- a follow-up time,
- and an event indicator.

These fields are available in the individual-level record.

In the Progenetix data model, one individual may have more than one biosample.  
Therefore, after merging biosample- and individual-level metadata, we also need to decide how to handle the one-to-many relationship before building a survival model.

In [ ]:
individual_ids = (
    biosample_df["individual_id"]
    .dropna()
    .astype(str)
    .unique()
    .tolist()
)

print("Number of unique individuals:", len(individual_ids))
print(individual_ids[:10])

In [ ]:
SESSION = requests.Session()
SESSION.headers.update({
    "User-Agent": "Mozilla/5.0",
    "Accept": "application/json, text/plain, */*",
})

def fetch_individual_record(individual_id, session=SESSION, timeout=120):
    # Fetch one Progenetix individual record by individual ID.
    url = f"https://progenetix.org/beacon/individuals/{individual_id}/"
    response = session.get(url, timeout=timeout)
    response.raise_for_status()
    return response.json()

In [ ]:
def parse_individual_record(data):
    # Parse one Progenetix individual JSON response into a flat dictionary.
    response = data.get("response", {})
    result_sets = response.get("resultSets", [])
    if not result_sets:
        return None

    results = result_sets[0].get("results", [])
    if not results:
        return None

    rec = results[0]

    diseases = rec.get("diseases", [])
    disease_0 = diseases[0] if len(diseases) > 0 else {}

    out = {
        # identifiers
        "individual_id": rec.get("id"),
        "n_diseases": len(diseases),

        # sex
        "sex_id_individual": rec.get("sex", {}).get("id"),
        "sex_label_individual": rec.get("sex", {}).get("label"),

        # top-level survival-related status
        "vital_status": rec.get("vitalStatus", {}).get("status"),

        # top-level info block
        "info_age_at_diagnosis_days": rec.get("info", {}).get("ageAtDiagnosis"),
        "info_days_to_death": rec.get("info", {}).get("daysToDeath"),
        "info_death": rec.get("info", {}).get("death"),
        "info_ethnicity": rec.get("info", {}).get("ethnicity"),
        "info_race": rec.get("info", {}).get("race"),
        "info_year_of_birth": rec.get("info", {}).get("yearOfBirth"),

        # references
        "tcga_case_id": rec.get("references", {}).get("tcgacase", {}).get("id"),
        "tcga_submitter_id": rec.get("references", {}).get("tcgasubmitter", {}).get("id"),

        # disease block 
        "disease_id": disease_0.get("diseaseCode", {}).get("id"),
        "disease_label": disease_0.get("diseaseCode", {}).get("label"),
        "followup_days": disease_0.get("followupDays"),
        "followup_time_iso": disease_0.get("followupTime"),
        "followup_state_id": disease_0.get("followupState", {}).get("id"),
        "followup_state_label": disease_0.get("followupState", {}).get("label"),
        "onset_age_iso": disease_0.get("onset", {}).get("age"),
        "onset_age_days": disease_0.get("onset", {}).get("ageDays"),
        "stage_id_individual": disease_0.get("stage", {}).get("id"),
        "stage_label_individual": disease_0.get("stage", {}).get("label"),
    }

    return out

In [ ]:
individual_rows = []
failed_individuals = []

for individual_id in tqdm(individual_ids, desc="Fetching individual records"):
    try:
        data = fetch_individual_record(individual_id)
        row = parse_individual_record(data)
        if row is not None:
            individual_rows.append(row)
        else:
            failed_individuals.append((individual_id, "No result"))
    except Exception as e:
        failed_individuals.append((individual_id, str(e)))

individual_df = pd.DataFrame(individual_rows)

print("Fetched individual rows:", len(individual_df))
print("Failed individual queries:", len(failed_individuals))
individual_df.head()

### Merge Biosample and Individual with individual_id

In [ ]:
merged_meta_df = biosample_df.merge(
    individual_df,
    on="individual_id",
    how="left"
)

print("Merged metadata shape:", merged_meta_df.shape)
merged_meta_df.head()

### Build Survival Outcome
Keep one biosample for each individual

In [ ]:
survival_df = merged_meta_df.copy()

# Survival time
survival_df["time"] = pd.to_numeric(survival_df["followup_days"], errors="coerce")

def map_event(row):
    """
    Convert individual-level follow-up / vital status into a binary event label.
    0 = alive / censored
    1 = dead / event occurred
    """
    followup_state = str(row.get("followup_state_label", "")).strip().lower()
    vital_status = str(row.get("vital_status", "")).strip().upper()

    if "alive" in followup_state:
        return 0
    if "deceased" in followup_state or "dead" in followup_state:
        return 1

    if vital_status == "ALIVE":
        return 0
    if vital_status == "DECEASED":
        return 1

    return np.nan

survival_df["event"] = survival_df.apply(map_event, axis=1)

# Keep only rows with usable survival data
survival_df = survival_df.dropna(subset=["biosample_id", "individual_id", "time", "event"]).copy()
survival_df["event"] = survival_df["event"].astype(int)

print("Rows with usable survival data:", len(survival_df))
survival_df[[
    "biosample_id",
    "individual_id",
    "time",
    "event",
    "followup_state_label",
    "vital_status",
    "sex_label_individual"
]].head()

In [ ]:
survival_unique_df = (
    survival_df
    .groupby("individual_id")
    .sample(n=1, random_state=42)   
    .copy()
)

print("Rows after keeping one biosample per individual:", len(survival_unique_df))
survival_unique_df[[
    "biosample_id",
    "individual_id",
    "sample_origin_type_label",
    "time",
    "event",
    "sex_label_individual"
]].head()

In [ ]:
FINAL_META_COLUMNS = [
    # biosample-level
    "biosample_id",
    "individual_id",
    "biosample_name",
    "notes",
    "histological_diagnosis_id",
    "histological_diagnosis_label",
    "pathological_stage_id",
    "pathological_stage_label",
    "sample_origin_type_id",
    "sample_origin_type_label",
    "sampled_tissue_id",
    "sampled_tissue_label",
    "tcgaproject_id",
    "tcgaproject_label",
    "icdo_morphology_id",
    "icdo_topography_id",
    "icdo_topography_label",

    # individual-level
    "sex_id_individual",
    "sex_label_individual",
    "vital_status",
    "info_age_at_diagnosis_days",
    "info_days_to_death",
    "info_death",
    "info_ethnicity",
    "info_race",
    "info_year_of_birth",
    "tcga_case_id",
    "tcga_submitter_id",
    "disease_id",
    "disease_label",
    "followup_days",
    "followup_time_iso",
    "followup_state_id",
    "followup_state_label",
    "onset_age_iso",
    "onset_age_days",
    "stage_id_individual",
    "stage_label_individual",

    # derived survival variables
    "time",
    "event",
]

FINAL_META_COLUMNS = [c for c in FINAL_META_COLUMNS if c in survival_unique_df.columns]

survival_unique_df = survival_unique_df[FINAL_META_COLUMNS].copy()

print("Final survival metadata shape:", survival_unique_df.shape)
survival_unique_df.head()

## Query gene-level CNV features with boolean Beacon requests

To keep the feature engineering simple, we use Boolean queries at the biosample level.

The idea is:

- `GENE_hit`: does this biosample contain at least one CNV event affecting the target gene?
- `GENE_gain_hit`: does this biosample contain at least one gain-type CNV event affecting the target gene?
- `GENE_loss_hit`: does this biosample contain at least one loss-type CNV event affecting the target gene?

With `requestedGranularity=boolean`, the Beacon endpoint returns a simple existence answer.


In [ ]:
TARGET_GENE = "EGFR"
print("Target gene:", TARGET_GENE)

## Define Boolean query functions

We now define helper functions that query the Progenetix biosample-scoped CNV endpoint.

For each biosample, we query three Boolean features:

1. any CNV hit affecting the gene,
2. any gain event affecting the gene,
3. any loss event affecting the gene.

The returned value is converted into a binary feature:
- `1` if at least one matching event exists,
- `0` otherwise.

In [ ]:
SESSION = requests.Session()
SESSION.headers.update({
    "User-Agent": "Mozilla/5.0",
    "Accept": "application/json, text/plain, */*",
})

def query_biosample_boolean(
    biosample_id,
    gene_symbol,
    extra_params=None,
    session=SESSION,
    timeout=120
):
    url = f"https://progenetix.org/beacon/biosamples/{biosample_id}/g_variants/"

    params = {
        "geneId": gene_symbol,
        "requestedGranularity": "boolean",
    }

    if extra_params:
        params.update(extra_params)

    response = session.get(url, params=params, timeout=timeout)
    response.raise_for_status()
    data = response.json()

    exists = data.get("responseSummary", {}).get("exists", False)
    return int(bool(exists))

## Define simplified Boolean CNV states

To keep the workflow simple, we collapse CNV states into two broader categories:

- **gain** = low-level gain or high-level gain
- **loss** = low-level loss or high-level loss

This allows us to define three compact Boolean features for the target gene:

- `GENE_hit`
- `GENE_gain_hit`
- `GENE_loss_hit`

This simplified design is easier to interpret than keeping separate high- and low-level states.

In [ ]:
# CNV state groups
GAIN_STATE_IDS = ["EFO:0030071", "EFO:0030072"]   # low-level gain, high-level gain
LOSS_STATE_IDS = ["EFO:0030068", "EFO:0020073"]   # low-level loss, high-level loss

print("GAIN_STATE_IDS =", GAIN_STATE_IDS)
print("LOSS_STATE_IDS =", LOSS_STATE_IDS)

In [ ]:
def make_gene_hit_params():
    # Boolean query for any CNV event affecting the target gene.
    return {}


def make_gene_gain_param_list():
    # Boolean query candidates for gain-related events.
    # One query per gain state ID.
    return [{"filters": state_id} for state_id in GAIN_STATE_IDS]


def make_gene_loss_param_list():
    # Boolean query candidates for loss-related events.
    # One query per loss state ID.
    return [{"filters": state_id} for state_id in LOSS_STATE_IDS]

def query_biosample_boolean(
    biosample_id,
    gene_symbol,
    extra_params=None,
    session=SESSION,
    timeout=120
):
    # Run a Boolean Beacon query against one biosample-scoped g_variants endpoint.
    url = f"https://progenetix.org/beacon/biosamples/{biosample_id}/g_variants/"

    params = {
        "geneId": gene_symbol,
        "requestedGranularity": "boolean",
    }

    if extra_params:
        params.update(extra_params)

    response = session.get(url, params=params, timeout=timeout)
    response.raise_for_status()
    data = response.json()

    exists = data.get("responseSummary", {}).get("exists", False)
    return int(bool(exists))

In [ ]:
def query_any_of_param_list(biosample_id, gene_symbol, param_list):
    # Return 1 if any Boolean query in param_list returns True, else 0.
    for params in param_list:
        try:
            hit = query_biosample_boolean(
                biosample_id=biosample_id,
                gene_symbol=gene_symbol,
                extra_params=params
            )
            if hit == 1:
                return 1
        except Exception:
            continue
    return 0

In [ ]:
def build_boolean_gene_features(
    biosample_ids,
    gene_symbol
):
    """
    Build three Boolean CNV features for one gene:
    - gene hit
    - gene gain hit
    - gene loss hit
    """
    rows = []
    failures = []

    gain_param_list = make_gene_gain_param_list()
    loss_param_list = make_gene_loss_param_list()

    for biosample_id in tqdm(biosample_ids, desc=f"Querying Boolean CNV features for {gene_symbol}"):
        try:
            gene_hit = query_biosample_boolean(
                biosample_id=biosample_id,
                gene_symbol=gene_symbol,
                extra_params=make_gene_hit_params()
            )

            gene_gain_hit = query_any_of_param_list(
                biosample_id=biosample_id,
                gene_symbol=gene_symbol,
                param_list=gain_param_list
            )

            gene_loss_hit = query_any_of_param_list(
                biosample_id=biosample_id,
                gene_symbol=gene_symbol,
                param_list=loss_param_list
            )

            rows.append({
                "biosample_id": biosample_id,
                f"{gene_symbol}_hit": gene_hit,
                f"{gene_symbol}_gain_hit": gene_gain_hit,
                f"{gene_symbol}_loss_hit": gene_loss_hit,
            })

        except Exception as e:
            failures.append((biosample_id, str(e)))
            rows.append({
                "biosample_id": biosample_id,
                f"{gene_symbol}_hit": np.nan,
                f"{gene_symbol}_gain_hit": np.nan,
                f"{gene_symbol}_loss_hit": np.nan,
            })

    feature_df = pd.DataFrame(rows)
    return feature_df, failures

In [ ]:
TARGET_GENE = "EGFR"

biosample_ids_for_features = survival_unique_df["biosample_id"].dropna().astype(str).tolist()

gene_feature_df, gene_feature_failures = build_boolean_gene_features(
    biosample_ids=biosample_ids_for_features,
    gene_symbol=TARGET_GENE,
)

print("Feature table shape:", gene_feature_df.shape)
print("Number of failed biosample queries:", len(gene_feature_failures))
gene_feature_df.head()